RAG with LLMs(Grok)

In [1]:
# import the necessary libraries 
import os
from dotenv import load_dotenv,find_dotenv
#find the path of the .env file
dotenv_path=find_dotenv()
#load the environment variables   
load_dotenv(dotenv_path)
#set the API key
api_key=os.getenv("GROQ_API_KEY")


# Data Ingestion Pipeline







In [2]:
#imports:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\SAHIL\AppData\Local\Temp\ipykernel_30172\3155417010.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader
c:\Users\SAHIL\OneDrive\Desktop\MY_RAG\Rag_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from unittest import loader
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs



In [4]:
all_pdf_documents = load_all_pdfs()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KHIJFK+MundoSans', '/Encoding': '/MacRomanEncoding', '/FirstChar': 33, '/FontDescriptor': IndirectObject(7203, 0, 2332867978848), '/LastChar': 223, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [349, 0, 0, 0, 700, 0, 0, 285, 285, 365, 516, 210, 381, 210, 372, 516, 516, 516, 516, 516, 516, 516, 516, 516, 516, 210, 210, 0, 516, 0, 421, 0, 606, 559, 608, 699, 488, 478, 661, 718, 274, 274, 568, 458, 891, 736, 729, 533, 720, 566, 475, 503, 677, 600, 914, 0, 533, 547, 235, 0, 235, 0, 0, 0, 478, 561, 451, 554, 489, 281, 490, 549, 242, 244, 465, 236, 838, 551, 562, 556, 559, 344, 374, 309, 540, 465, 716, 440, 471, 426, 0, 0, 0, 0, 236, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 562, 0, 0, 0, 0, 0, 0, 0, 0, 516, 0, 0, 0, 0, 0, 0, 0, 0, 0, 236, 0, 0, 236, 0, 236, 236, 0, 0, 236, 236, 236, 236, 236, 0, 0, 236, 0, 0, 0, 0, 0, 236, 0, 236, 236, 0, 0, 0, 0,

total pdfs: 3
total pages: 1159


#Chucking:

In [5]:
from lzma import CHECK_UNKNOWN
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=1000,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs



In [6]:
chunks = split_docs(all_pdf_documents)

In [7]:
len(chunks)

2176

# Embedding Model 

In [8]:
import sys
print(sys.executable)

c:\Users\SAHIL\OneDrive\Desktop\MY_RAG\Rag_project\venv\Scripts\python.exe


In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1818.25it/s]


In [11]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings


In [12]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1444.38it/s]


embedding dimensions= 384


C:\Users\SAHIL\AppData\Local\Temp\ipykernel_30172\2000907449.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


# Vector Database( chromaDB)

In [13]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
    def add_documents(self, documents, embeddings):
       if len(documents)!= len(embeddings):
        raise ValueError("num of documents does not match num of embeddings")

    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
        doc_id = f"doc_{uuid.uuid4()}"
        ids.append(doc_id)

        metadata = dict(doc.metadata)
        metadata["doc_index"] = i
        metadata["content_length"] = len(doc.page_content)

        all_metadata.append(metadata)
        documents_content.append(doc.page_content)
        embeddings_list.append(embedding.tolist())

    # Add everything ONCE
    self.collection.add(
        ids=ids,
        metadatas=all_metadata,
        documents=documents_content,
        embeddings=embeddings_list
    )

    print("total documents added in vector store =", len(documents_content))
    print("docs in collection:", self.collection.count())
    

IndentationError: expected an indented block after function definition on line 24 (675936503.py, line 25)

In [ ]:
import os
import chromadb

In [ ]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [ ]:

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches: 100%|██████████| 68/68 [01:42<00:00,  1.51s/it]


embeddings shape: (2176, 384)
total documents added in vector store= 2176
docs in collection: 2980


# Retrieval Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever.retrieve("What is eigen value ?")

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.57it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_d146cc54-8e9b-4d1a-988c-165463485ce4',
  'document': 'the value of this scalar, λ, the eigenvalue.\n(continued...)',
  'metadata': {'ebx_publisher': 'Oxford University Press USA',
   'moddate': '2014-06-28T09:57:50+01:00',
   'page_label': '493',
   'page': 501,
   'total_pages': 617,
   'producer': 'Acrobat Distiller 8.1.0 (Windows)',
   'content_length': 59,
   'author': 'Kuldeep Singh',
   'doc_index': 998,
   'source': 'data\\Linear AlgebraStep by Step Kuldeep Singh.pdf',
   'title': 'Linear Algebra: Step by Step',
   'creationdate': '2014-01-02T04:25:00+00:00',
   'creator': 'PyPDF'},
  'distance': 0.6490678787231445,
  'similarity_score': 0.35093212127685547,
  'rank': 1},
 {'id': 'doc_891cd98f-0f15-45dd-ac96-f92340a17ebb',
  'document': 'important question is how are the eigenvalues of a matrix identified?\nFinding Eigenvalues | 217',
  'metadata': {'content_length': 95,
   'icnappversion': '2.03',
   'source': 'data\\Practical Linear Algebra for Data Science (Cohen

In [ ]:
# groq LLMs:
#%pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
#LLMs
llm = ChatGroq(
    groq_api_key= api_key,
    model="groq/compound-mini",
    temperature=0.1,
    max_tokens=1024
)

In [ ]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is", rag_retriever, llm)

Batches: 100%|██████████| 1/1 [00:00<00:00, 15.25it/s]


embeddings shape: (1, 384)
retrieved 3 documents


In [ ]:
print(answer)

A **vector space** (also called a linear space) is a collection \(V\) of objects called *vectors* together with two operations:

1. **Vector addition**: for any two vectors \(u, v \in V\) there is a sum \(u+v \in V\);
2. **Scalar multiplication**: for any scalar \(a\) from a field (e.g., the real numbers \(\mathbb{R}\) or complex numbers \(\mathbb{C}\)) and any vector \(v \in V\) there is a product \(a v \in V\).

These operations must satisfy the usual axioms (associativity, commutativity of addition, existence of a zero vector, additive inverses, distributivity of scalar multiplication over vector addition and over field addition, compatibility of scalar multiplication with field multiplication, and \(1\cdot v = v\)).

When a vector space is non‑zero, any basis—a set of linearly independent vectors that spans the space—has the same number of elements; that number is called the **dimension** of the vector space. Classic examples of finite‑dimensional vector spaces are the Euclidean sp